In [2]:
import argparse
import json
import re
import sys
from py2neo import Graph, Node, Relationship, NodeMatcher, cypher


In [43]:
# -----------------------------------------------------------------
# BUILD_LABEL
# -----------------------------------------------------------------
def build_label(txt):
    if txt.startswith('intrusion-set'):
        return 'Group'
    if txt.startswith('malware'):
        return 'Software'
    if txt.startswith('tool'):
        return 'Tool'
    if txt.startswith('attack-pattern'):
        return 'Technique'
    if txt.startswith('course-of-action'):
        return 'Mitigation'
    if txt.startswith('x-mitre-tactic'):
        return 'Tactic'
    return 'Unknown'

In [60]:
# -----------------------------------------------------------------
# BUILD ALIASES
# -----------------------------------------------------------------
def build_objects(obj,key):
    label = build_label(obj['type'])
    
    # add properties
    props = {}
    props['name'] = obj['name']
    props['id'] = obj['id']
    props['type'] = obj['type']
    if obj.get('description'):
        props['description'] = obj['description'] # cypher.cypher_escape( obj['description'] )
    if obj.get('created'):
        props['created'] = obj['created']
    if obj.get('modified'):
        props['modified'] = obj['modified']
    if obj.get('x_mitre_version'):
        props['version'] = obj['x_mitre_version']
    #if obj.get('external_references'):
    #    props['external_id'] = obj['external_references']['external_id']
    # create node for the group
    node_main = Node(label, **props)
    # merge node to graph
    graph.merge(node_main,label,'name')
    
    # dealing with aliases
    if obj.get('aliases'): 
        aliases = obj['aliases']
    elif obj.get('x_mitre_aliases'): 
        aliases = obj['x_mitre_aliases']
    else:
        aliases = None
    
    if aliases:
        for alias in aliases:
            if alias != obj['name']:
                node_alias = Node('Alias', name=alias, type=obj['type'])
                relation = Relationship.type('alias')
                graph.merge(relation(node_main,node_alias),label,'name')

In [61]:
# -----------------------------------------------------------------
# BUILD RELATIONS
# -----------------------------------------------------------------
def build_relations(obj):
    if not gnames.get(obj['source_ref']): return
    if not gnames.get(obj['target_ref']): return
    
    
    m = NodeMatcher(graph)
    
    source = m.match( build_label(obj['source_ref']), name=gnames[obj['source_ref']] ).first()
    target = m.match( build_label(obj['target_ref']), name=gnames[obj['target_ref']] ).first()
    
    # source = Node( build_label(obj['source_ref']), name=gnames[obj['source_ref']], id=obj['source_ref'] )
    # target = Node( build_label(obj['target_ref']), name=gnames[obj['target_ref']], id=obj['target_ref'] )
    relation = Relationship.type( obj['relationship_type'] )
    
    graph.merge(relation(source,target),build_label(obj['source_ref']),'name')
    # print('Relation: "%s" -[%s]-> "%s"' % (gnames[obj['source_ref']],obj['relationship_type'],gnames[obj['target_ref']]) )
    

In [62]:
# checks arguments and options
#dbg_mode = True if args.debug else None
json_file = "./enterprise-attack.json"
# load JSON data from file
try:
    with open(json_file) as fh:
        data = json.load(fh)
    fh.close()

except Exception as e:
    sys.stderr.write( '[ERROR] reading configuration file %s\n' % json_file )
    sys.stderr.write( '[ERROR] %s\n' % str(e) )
    sys.exit(1)

In [63]:
# open graph connection
graph_bolt = "bolt://127.0.0.1:7687"
graph_auth = ("neo4j","ZlatanisRed20#")

graph = Graph(graph_bolt,auth=graph_auth)

# Delete existing nodes and edges
graph.delete_all()

In [64]:
# Global names
gnames = {}

# Walk through JSON objects to create nodes
for obj in data['objects']:
    # if JSON object is about Groups
    if obj['type']=='intrusion-set':
        gnames[ obj['id'] ] = obj['name']
        build_objects(obj,'groups')
    
    # if JSON object is about Softwares
    if obj['type']=='malware':
        gnames[ obj['id'] ] = obj['name']
        build_objects(obj,'malware')
        
    # if JSON object is about Tools
    if obj['type']=='tool':
        gnames[ obj['id'] ] = obj['name']
        build_objects(obj,'tool')
    
    # if JSON object is about Techniques
    if obj['type']=='attack-pattern':
        gnames[ obj['id'] ] = obj['name']
        build_objects(obj,'Technique')
        
    # course-of-action/ Mitigation
    if obj['type']=='course-of-action':
        gnames[ obj['id'] ] = obj['name']
        build_objects(obj, 'Mitigation')
    
    # mitre tactics
    if obj['type'] == 'x-mitre-tactic':
        gnames[ obj['id'] ] = obj['name']
        build_objects(obj, 'Tactic')
        

In [65]:
# Walk through JSON objects to create edges
for obj in data['objects']:
    # if JSON object is about Relationships
    if obj['type']=='relationship':
        build_relations(obj)

In [66]:
object_types = set()
for obj in data['objects']:
    object_types.add(obj['type'])
print(object_types)

{'x-mitre-tactic', 'marking-definition', 'intrusion-set', 'tool', 'course-of-action', 'x-mitre-matrix', 'identity', 'relationship', 'attack-pattern', 'malware'}


In [54]:
print(gnames)

{'attack-pattern--01df3350-ce05-4bdf-bdf8-0a919a66d4a8': '.bash_profile and .bashrc', 'attack-pattern--dcaa092b-7de9-4a21-977f-7fcb77e89c48': 'Access Token Manipulation', 'attack-pattern--9b99b83a-1aac-4e29-b975-b374950551a3': 'Accessibility Features', 'attack-pattern--b24e2a20-3b3d-4bf0-823b-1ed765398fb0': 'Account Access Removal', 'attack-pattern--72b74d71-8169-42aa-92e0-e7b04b9f5a08': 'Account Discovery', 'attack-pattern--a10641f4-87b4-45a3-a906-92a149cb2c27': 'Account Manipulation', 'attack-pattern--4bf5845d-a814-4490-bc5c-ccdee6043025': 'AppCert DLLs', 'attack-pattern--317fefa6-46c7-4062-adb6-2008cf6bcb41': 'AppInit DLLs', 'attack-pattern--5ad95aaa-49c1-4784-821d-2e83f47b079b': 'AppleScript', 'attack-pattern--27960489-4e7f-461d-a62a-f5c0cb521e4a': 'Application Access Token', 'attack-pattern--327f3cc5-eea1-42d4-a6cd-ed34b7ce8f61': 'Application Deployment Software', 'attack-pattern--7c93aa74-4bc0-4a9e-90ea-f25f86301566': 'Application Shimming', 'attack-pattern--4ae4f953-fe58-4cc8-a3

In [ ]:
# x-mitre-tactic: [name, description, created_by_ref, created, id, x_mitre-shortname, external_references [external_id, u=rl]
# marking-definition: no good 
# intrusion-set - APT groups likely
# tool 
# course-of-action - mitigation 
# x-mitre-matrix - matrix info 
# identity - mitre stuff
# relationship, types = mitigates, uses, revoked-by
# attack_pattern - data sources, name, description, id, platform, kill-chain pahses[{[phase_name]}], external_references[external_id, url]

# preparing training data for statistical causation/correlation - MITRE ATT&CK APT groups, and tools so far
# working on incorporation IOCs in the equation 
